In [1]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
import os
import sys
import pickle
import random
import torch
import torch.nn.functional as F
import torchvision.transforms as tfms
import numpy as np
from PIL import Image


from diffusers import (
    StableDiffusionPipeline,
    DPMSolverMultistepScheduler,
)

SPC_ROOT = "/content/SPC"

# Dataset produced by the original pipeline
DATASETS_ROOT = f"{SPC_ROOT}/datasets-by-seed"

# Output
OUTPUT_ROOT = f"{SPC_ROOT}/"

# Experiment
dataset = "eurosat"          # dtd / eurosat / cars / pets / flowers / fgvca
SEED = 2
STARTING_STEP = 15
BATCH_SIZE = 16    # Start with 1 or 2 on T4
CFG_STRENGTH = 8.0
IMAGES_PER_CLASS = 64         # Test first; change to 64 later
MIXUP = False
USE_LLAVA = True

# Models
SD_MODEL_NAME = "stable_diffusion"
SD_MODEL_ID = "sd2-community/stable-diffusion-2-1"


if SPC_ROOT not in sys.path:
    sys.path.insert(0, SPC_ROOT)

random.seed(22)
np.random.seed(22)
torch.manual_seed(22)

print("SPC:", SPC_ROOT)
print("Dataset:", dataset)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SPC: /content/SPC
Dataset: eurosat
CUDA: True
GPU: NVIDIA L4


## 2. Install dependencies

In [2]:
!pip install -q -U diffusers transformers accelerate bitsandbytes sentencepiece huggingface_hub

## 3. Import SPC and LLaVA modules

In [3]:
!git clone https://github.com/canhdinhtien/SPC.git

fatal: destination path 'SPC' already exists and is not an empty directory.


In [4]:
%cd /content/SPC

/content/SPC


In [5]:
!pip install munch

In [6]:
import sys
import os

from data import get_data_loader
from utils import fix_random_seeds, get_dataset_name_for_template
from util_data import SUBSET_NAMES, TEMPLATES_SMALL
import torch
import os
import sys

PROJECT_ROOT = "/content/SPC"

## 4. Image and latent utilities

In [7]:
def pad_image(image):
    width, height = image.size
    if width > height:
        new_width = new_height = width
    else:
        new_width = new_height = height

    new_im = Image.new("RGB", (new_width, new_height))
    new_im.paste(
        image,
        ((new_width - width) // 2, (new_height - height) // 2)
    )
    return new_im.resize((768, 768))


def load_image(path):
    return pad_image(Image.open(path).convert("RGB"))


@torch.no_grad()
def pil_to_latents(image, vae):
    x = tfms.ToTensor()(image).unsqueeze(0) * 2.0 - 1.0
    x = x.to("cuda", dtype=torch.float16)
    return vae.encode(x).latent_dist.sample() * 0.18215


@torch.no_grad()
def path_to_latents(paths, vae, mixup=False):
    images = [load_image(p) for p in paths]

    if mixup:
        lambdas = np.random.beta(0.2, 0.2, size=len(images))
        mixed_images = []
        for i in range(len(images)):
            i1 = random.randint(0, len(images) - 1)
            i2 = random.randint(0, len(images) - 1)
            mixed_images.append(
                Image.blend(images[i1], images[i2], lambdas[i])
            )
        images = mixed_images

    return torch.cat([pil_to_latents(img, vae) for img in images])


@torch.no_grad()
def latents_to_pil(latents, vae):
    latents = latents / 0.18215
    image = vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
    image = (image * 255).round().astype("uint8")
    return [Image.fromarray(x) for x in image]


@torch.no_grad()
def text_enc(prompts, tokenizer, text_encoder, maxlen=None):
    if maxlen is None:
        maxlen = tokenizer.model_max_length

    inp = tokenizer(
        prompts,
        padding="max_length",
        max_length=maxlen,
        truncation=True,
        return_tensors="pt",
    )
    return text_encoder(inp.input_ids.to("cuda"))[0].half()


## 5. Load Stable Diffusion

In [8]:
def load_stable_diffusion(model_name):
    print("Loading Stable Diffusion...")

    model_id = (
        SD_MODEL_ID
        if model_name == "stable_diffusion"
        else "SG161222/Realistic_Vision_V2.0"
    )

    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
    )
    pipe = pipe.to("cuda")

    scheduler = DPMSolverMultistepScheduler.from_config(
        pipe.scheduler.config
    )
    scheduler.set_timesteps(20)
    pipe.scheduler = scheduler

    print("Stable Diffusion loaded.")
    return (
        pipe.vae,
        pipe.unet,
        pipe.scheduler,
        pipe.tokenizer,
        pipe.text_encoder,
    )


In [9]:
class_names = SUBSET_NAMES[dataset]
class_names

['Annual Crop Land',
 'Forest',
 'Herbaceous Vegetation Land',
 'Highway or Road',
 'a Industrial Building',
 'Pasture Land',
 'Permanent Crop Land',
 'Residential Building',
 'River',
 'Sea or Lake']

## 11. Load Stable Diffusion and LLaVA

In [10]:
vae, unet, scheduler, tokenizer, text_encoder = (
    load_stable_diffusion(SD_MODEL_NAME)
)
print("Generation models loaded.")


Loading Stable Diffusion...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

Stable Diffusion loaded.
Generation models loaded.


In [11]:
def get_filtered_paths(dataset_name, n_samples_per_class, real_data_dir):
    print(f"Lọc {n_samples_per_class}-shot dữ liệu gốc từ {real_data_dir}...")
    train_loader, _ = get_data_loader(
        real_train_data_dir=real_data_dir,
        real_test_data_dir=real_data_dir,
        dataset=dataset_name,
        bs=1, eval_bs=1,
        n_img_per_cls=n_samples_per_class,
        model_type="qwen"
    )

    train_dataset = train_loader.dataset
    class_names = SUBSET_NAMES[dataset_name]
    paths_dict = {cls: [] for cls in class_names}

    if dataset_name in ['dtd', 'flowers102', 'food101', 'sun397', 'fgvc_aircraft']:
        _images = train_dataset._image_files
        _labels = train_dataset._labels
    elif dataset_name == 'eurosat':
        _images = [sample[0] for sample in train_dataset.samples]
        _labels = [sample[1] for sample in train_dataset.samples]
    elif dataset_name == 'pets':
        _images = train_dataset._images
        _labels = train_dataset._labels
    elif dataset_name == 'cars':
        _images = [sample[0] for sample in train_dataset._samples]
        _labels = [sample[1] for sample in train_dataset._samples]
    elif dataset_name == 'caltech101':
        _images = [
            os.path.join(
                train_dataset.root,
                "caltech101",
                "101_ObjectCategories",
                train_dataset.categories[train_dataset.y[i]],
                f"image_{train_dataset.index[i]:04d}.jpg"
            ) for i in range(len(train_dataset.index))
        ]
        _labels = train_dataset.y
    else:
        raise ValueError(f"Dataset {dataset_name} not supported.")

    for img_path, label in zip(_images, _labels):
        class_name = class_names[label]
        paths_dict[class_name].append(img_path)

    return paths_dict, class_names

In [12]:
n_samples = 16
real_data_dir = "/content/SPC/data"
paths_dict, class_names = get_filtered_paths(dataset, n_samples, real_data_dir)

Lọc 16-shot dữ liệu gốc từ /content/SPC/data...
16


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [13]:

class_names[:10]

['Annual Crop Land',
 'Forest',
 'Herbaceous Vegetation Land',
 'Highway or Road',
 'a Industrial Building',
 'Pasture Land',
 'Permanent Crop Land',
 'Residential Building',
 'River',
 'Sea or Lake']

In [14]:
out_root = "/content"

In [15]:
DATA_ROOT = "/content/SPC/data/train/eurosat/2750"

In [16]:
class_names

['Annual Crop Land',
 'Forest',
 'Herbaceous Vegetation Land',
 'Highway or Road',
 'a Industrial Building',
 'Pasture Land',
 'Permanent Crop Land',
 'Residential Building',
 'River',
 'Sea or Lake']

In [17]:
paths_dict['Annual Crop Land']

['/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1537.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1917.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1089.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_57.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_2651.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1696.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_9.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1456.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_720.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_23.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1292.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1873.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_2019.jpg',
 '/content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1182.jpg',


In [18]:
import os
import glob
import random

# Sửa đường dẫn này theo dataset của bạn
TEST_CLASS = 'Annual Crop Land'

TEST_DIR = os.path.join(
    DATA_ROOT,
    TEST_CLASS
)

image_extensions = (
    "*.jpg",
    "*.jpeg",
    "*.png",
    "*.JPG",
    "*.JPEG",
    "*.PNG",
)

test_paths = []

for ext in image_extensions:
    test_paths.extend(
        glob.glob(
            os.path.join(TEST_DIR, ext)
        )
    )

random.seed(42)
test_paths = paths_dict[TEST_CLASS]
random.shuffle(test_paths)

# Chỉ lấy 8 ảnh
test_paths = test_paths[:8]

print("Number of test images:", len(test_paths))

for i, path in enumerate(test_paths):
    print(f"[{i}] {path}")

Number of test images: 8
[0] /content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1456.jpg
[1] /content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_23.jpg
[2] /content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1696.jpg
[3] /content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_9.jpg
[4] /content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_2208.jpg
[5] /content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_1292.jpg
[6] /content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_2019.jpg
[7] /content/SPC/data/train/eurosat/2750/AnnualCrop/AnnualCrop_720.jpg


In [19]:
import pickle

path = "/content/issynth_eurosat.pkl"

with open(path, "rb") as f:
    issynth_prompts = pickle.load(f)

In [20]:
class_names

['Annual Crop Land',
 'Forest',
 'Herbaceous Vegetation Land',
 'Highway or Road',
 'a Industrial Building',
 'Pasture Land',
 'Permanent Crop Land',
 'Residential Building',
 'River',
 'Sea or Lake']

In [21]:
class_names_issynth = issynth_prompts.keys()
class_names_issynth

dict_keys(['annual crop land', 'forest', 'herbaceous vegetation land', 'highway or road', 'industrial building', 'pasture land', 'permanent crop land', 'residential building', 'river', 'sea or lake'])

In [22]:
issynth_prompts['annual crop land']

['a crop of wheat is harvested from the ground annually .',
 'a land filled with many annual crops',
 'the country saw huge acreage of cropping land during their annual planting campaign',
 'a map of the country showing the land covered in annual crops',
 'annual crop cultivated at pasture and cleared land in the mountains',
 "aerial drone footage of crops being cultivated in the country's rural coastal countryside during the annual harvest",
 'crop on agricultural land from which is produced every year',
 'farm in an ideal spot to land the annual crop .',
 'crop is the biggest crop we have ever planted and has raised to date totals of acres .',
 'the annual crop has been planted for an area of rural land .',
 'a man demonstrates how to land a ripe annual crop',
 'a man growing crop on a pond in the year over land',
 'an invasive species is cultivated all year long and lays waste on agricultural land',
 'crop on land is the most important agricultural crop of the year .',
 'growing cro

In [23]:
idx_to_name = {
    i: name
    for i, name in enumerate(class_names_issynth)
}

In [24]:
idx_to_name[0]

'annual crop land'

In [25]:
import gc
import math
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


NUM_DENOISING_STEPS = 20

# Keep approximately 25% noise
# according to the original 20-step schedule
STARTING_STEP = 15



In [ ]:
# ============================================================
# GENERATION
# ============================================================

for idx, class_ in enumerate(class_names):

    class_dir = DATA_ROOT

    output_dir = os.path.join(
        out_root,
        class_
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    # --------------------------------------------------------
    # 1. Get available real images
    # --------------------------------------------------------

    available_files = paths_dict[class_]

    # --------------------------------------------------------
    # 2. Count already generated images
    # --------------------------------------------------------

    generated_this_class = len([
        x
        for x in os.listdir(output_dir)
        if x.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
                ".webp"
            )
        )
    ])

    # --------------------------------------------------------
    # 3. Check prompts for this class
    # --------------------------------------------------------

    class_prompts = issynth_prompts[idx_to_name[idx]]

    if len(class_prompts) == 0:
        print(f"WARNING: No prompts found for class {class_}")
        continue

    print(
        f"\n[{idx + 1}/{len(class_names)}] "
        f"{class_}: "
        f"{generated_this_class}/"
        f"{IMAGES_PER_CLASS} images"
    )

    print(
        f"Number of prompts for this class: "
        f"{len(class_prompts)}"
    )

    # --------------------------------------------------------
    # 4. If already enough images -> skip
    # --------------------------------------------------------

    if generated_this_class >= IMAGES_PER_CLASS:

        print(
            f"Already have {generated_this_class} images. "
            f"Skipping {class_}."
        )

        continue

    # --------------------------------------------------------
    # 5. Number of images still needed
    # --------------------------------------------------------

    remaining = (
        IMAGES_PER_CLASS
        - generated_this_class
    )

    print(
        f"Need to generate {remaining} more images."
    )

    # --------------------------------------------------------
    # 6. Cycle counter
    # --------------------------------------------------------

    cycles_spent_this_class = 0

    # --------------------------------------------------------
    # 7. Scheduler setup
    # --------------------------------------------------------

    scheduler.set_timesteps(
        NUM_DENOISING_STEPS
    )

    original_timesteps = (
        scheduler.timesteps.clone()
    )

    start_timestep = (
        original_timesteps[
            STARTING_STEP
        ].item()
    )

    print(
        "Original timesteps:",
        original_timesteps.tolist()
    )

    print(
        f"STARTING_STEP = {STARTING_STEP}"
    )

    print(
        f"Starting timestep = {start_timestep}"
    )

    # ========================================================
    # GENERATE UNTIL ENOUGH IMAGES
    # ========================================================

    while remaining > 0:

        cycles_spent_this_class += 1
        # ----------------------------------------------------
        # Actual batch size
        # ----------------------------------------------------
        current_batch_size = min(
            BATCH_SIZE,
            remaining
        )

        print(
            f"\n{'=' * 60}"
        )

        print(
            f"Class: {class_}"
        )

        print(
            f"Batch: {cycles_spent_this_class}"
        )

        print(
            f"Batch size: {current_batch_size}"
        )

        print(
            f"Progress: "
            f"{generated_this_class}/"
            f"{IMAGES_PER_CLASS}"
        )

        # ====================================================
        # 1. Sample real few-shot images
        # ====================================================

        file_names = random.choices(
            available_files,
            k=current_batch_size
        )

        file_paths = [
            os.path.join(
                class_dir,
                x
            )
            for x in file_names
        ]

        # ----------------------------------------------------
        # Real image -> latent
        # ----------------------------------------------------

        latents = path_to_latents(
            file_paths,
            vae,
            MIXUP
        )

        # ====================================================
        # 2. Add noise
        # ====================================================

        noise = torch.randn_like(
            latents
        )

        # One timestep per image
        timesteps = torch.full(
            (
                current_batch_size,
            ),
            start_timestep,
            device=latents.device,
            dtype=torch.long
        )

        noised_latents = scheduler.add_noise(
            latents,
            noise,
            timesteps
        )

        print(
            f"Added noise at timestep "
            f"{start_timestep}"
        )

        # ====================================================
        # 3. SELECT PROMPTS FOR THIS BATCH
        # ====================================================

        # Randomly choose one prompt for each image
        prompts = random.choices(
            class_prompts,
            k=current_batch_size
        )

        print(
            "\nPrompts used for generation:"
        )

        for i, prompt in enumerate(prompts):

            print(
                f"  [{i}] {repr(prompt)}"
            )

        # ====================================================
        # 4. Text embedding
        # ====================================================

        text_embed = text_enc(
            prompts,
            tokenizer,
            text_encoder
        )

        # ====================================================
        # 5. Unconditional embedding
        # ====================================================

        uncond = text_enc(
            [""] * current_batch_size,
            tokenizer,
            text_encoder,
            text_embed.shape[1]
        )

        # ====================================================
        # 6. CFG embedding
        # ====================================================

        emb = torch.cat(
            [
                uncond,
                text_embed
            ]
        )

        # ====================================================
        # 7. CREATE DENOISING TIMESTEPS
        # ====================================================

        denoising_timesteps = (
            torch.linspace(
                start_timestep,
                0,
                NUM_DENOISING_STEPS,
                device="cpu"
            )
            .round()
            .long()
        )

        # Remove duplicates
        denoising_timesteps = (
            torch.unique(
                denoising_timesteps,
                sorted=True
            )
            .flip(0)
        )

        print(
            "Number of denoising steps:",
            len(denoising_timesteps)
        )

        # ====================================================
        # 8. Set scheduler
        # ====================================================

        scheduler.set_timesteps(
            timesteps=denoising_timesteps,
            device=latents.device
        )

        # ====================================================
        # 9. DENOISING
        # ====================================================

        latents = noised_latents

        unet.to("cuda")

        for step_idx, ts in enumerate(
            scheduler.timesteps
        ):

            print(
                f"\rDenoising: "
                f"{step_idx + 1}/"
                f"{len(scheduler.timesteps)}",
                end=""
            )

            # ------------------------------------------------
            # CFG input
            # ------------------------------------------------

            inp = (
                scheduler.scale_model_input(
                    torch.cat(
                        [latents] * 2
                    ),
                    ts
                )
            )

            # ------------------------------------------------
            # UNet
            # ------------------------------------------------

            with torch.no_grad(), torch.autocast(
                "cuda"
            ):

                unconditional, conditional = (
                    unet(
                        inp,
                        ts,
                        encoder_hidden_states=emb
                    )
                    .sample
                    .chunk(2)
                )

            # ------------------------------------------------
            # Classifier-Free Guidance
            # ------------------------------------------------

            predicted_sample = (
                unconditional
                + CFG_STRENGTH
                * (
                    conditional
                    - unconditional
                )
            )

            # ------------------------------------------------
            # DPMSolver++
            # ------------------------------------------------

            latents = scheduler.step(
                predicted_sample,
                ts,
                latents
            ).prev_sample

        print()

        # ====================================================
        # 10. Move UNet to CPU
        # ====================================================

        unet.to("cpu")

        torch.cuda.empty_cache()

        # ====================================================
        # 11. Latent -> PIL
        # ====================================================

        final_imgs = latents_to_pil(
            latents,
            vae
        )

        # ====================================================
        # 12. Resize
        # ====================================================

        final_imgs_highres = [
            img.resize(
                (512, 512)
            )
            for img in final_imgs
        ]

        print(
            f"Generated "
            f"{len(final_imgs_highres)} "
            f"images in this batch."
        )

        # ====================================================
        # 13. SAVE IMAGES
        # ====================================================

        for i, img in enumerate(
            final_imgs_highres
        ):

            image_number = (
                generated_this_class
                + i
                + 1
            )

            save_path = os.path.join(
                output_dir,
                f"{class_}_{image_number:05d}.jpg"
            )

            img.save(
                save_path,
                quality=95
            )

        # ====================================================
        # 14. UPDATE COUNTER
        # ====================================================

        generated_this_class += (
            len(final_imgs_highres)
        )

        remaining = (
            IMAGES_PER_CLASS
            - generated_this_class
        )

        print(
            f"\nSaved batch successfully."
        )

        print(
            f"Progress: "
            f"{generated_this_class}/"
            f"{IMAGES_PER_CLASS}"
        )

        # ====================================================
        # 15. CLEAN GPU MEMORY
        # ====================================================

        del latents
        del noised_latents
        del noise
        del text_embed
        del uncond
        del emb

        gc.collect()

        torch.cuda.empty_cache()

    # ========================================================
    # CLASS FINISHED
    # ========================================================

    print(
        f"\n{'=' * 60}"
    )

    print(
        f"FINISHED CLASS: {class_}"
    )

    print(
        f"Generated: "
        f"{generated_this_class}/"
        f"{IMAGES_PER_CLASS}"
    )

    print(
        f"{'=' * 60}"
    )


# ============================================================
# Generation finished
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "GENERATION FINISHED"
)

print(
    "=" * 70
)


[1/10] Annual Crop Land: 64/64 images
Number of prompts for this class: 30
Already have 64 images. Skipping Annual Crop Land.

[2/10] Forest: 24/64 images
Number of prompts for this class: 30
Need to generate 40 more images.
Original timesteps: [999, 949, 899, 849, 799, 749, 699, 649, 599, 549, 500, 450, 400, 350, 300, 250, 200, 150, 100, 50]
STARTING_STEP = 15
Starting timestep = 250

Class: Forest
Batch: 1
Batch size: 16
Progress: 24/64
Added noise at timestep 250

Prompts used for generation:
  [0] 'a wild cherry tree growing among the trees and a forest'
  [1] 'the forest in autumn'
  [2] 'the forest in autumn'
  [3] 'dark forest and wild flowers of the forest'
  [4] 'a beautiful oak forest with green leaves'
  [5] 'a wild forest with snow and a bird .'
  [6] 'a forest surrounded by mountains'
  [7] 'a girl walks in a forest'
  [8] 'a forest is full of snow'
  [9] 'a bird spawns in the shady forest'
  [10] 'a tree in the forest'
  [11] 'a woman hiking through the forest'
  [12] 't

In [ ]:

import os
import matplotlib.pyplot as plt
from PIL import Image


def compare_image_folders(
    folder1,
    folder2,
    n_images=8,
    figsize_per_image=3,
    title1=None,
    title2=None
):

    extensions = (
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".bmp"
    )

    # =========================================================
    # Get image files
    # =========================================================

    files1 = sorted([
        f for f in os.listdir(folder1)
        if f.lower().endswith(extensions)
    ])

    files2 = sorted([
        f for f in os.listdir(folder2)
        if f.lower().endswith(extensions)
    ])

    print(f"{title1}: {len(files1)} images")
    print(f"{title2}: {len(files2)} images")

    # Number of images to display
    n = min(
        n_images,
        len(files1),
        len(files2)
    )

    if n == 0:
        print("Không tìm thấy ảnh.")
        return

    files1 = files1[:n]
    files2 = files2[:n]

    # =========================================================
    # Create 2 x d grid
    # =========================================================

    fig, axes = plt.subplots(
        2,
        n,
        figsize=(
            figsize_per_image * n,
            figsize_per_image * 2.2
        )
    )

    # =========================================================
    # Special case: n = 1
    # =========================================================

    if n == 1:
        axes = axes.reshape(2, 1)

    # =========================================================
    # Row 1: Folder 1
    # =========================================================

    for i, filename in enumerate(files1):

        path = os.path.join(
            folder1,
            filename
        )

        img = Image.open(path).convert("RGB")

        axes[0, i].imshow(img)

        axes[0, i].axis("off")

        # Chỉ ghi filename nhỏ bên dưới
        axes[0, i].set_title(
            filename,
            fontsize=9
        )

    # =========================================================
    # Row 2: Folder 2
    # =========================================================

    for i, filename in enumerate(files2):

        path = os.path.join(
            folder2,
            filename
        )

        img = Image.open(path).convert("RGB")

        axes[1, i].imshow(img)

        axes[1, i].axis("off")

        axes[1, i].set_title(
            filename,
            fontsize=9
        )

    # =========================================================
    # Row labels
    # =========================================================

    axes[0, 0].text(
        -0.08,
        0.5,
        title1,
        transform=axes[0, 0].transAxes,
        fontsize=14,
        fontweight="bold",
        rotation=90,
        va="center",
        ha="center"
    )

    axes[1, 0].text(
        -0.08,
        0.5,
        title2,
        transform=axes[1, 0].transAxes,
        fontsize=14,
        fontweight="bold",
        rotation=90,
        va="center",
        ha="center"
    )

    # =========================================================
    # Layout
    # =========================================================

    plt.tight_layout()

    plt.show()


In [ ]:
folder1 = "/content/Annual Crop Land"
folder2 = "/content/SPC/data/train/eurosat/2750/AnnualCrop"

In [ ]:
import os
import math
import matplotlib.pyplot as plt
from PIL import Image


def visualize_folder(
    folder,
    n_images=8,
    cols=4,
    figsize_per_image=3,
    random_sample=False,
    title=None
):
    """
    Visualize images from a folder.

    Parameters
    ----------
    folder : str
        Đường dẫn thư mục.

    n_images : int
        Số ảnh muốn hiển thị.

    cols : int
        Số cột muốn hiển thị.

    figsize_per_image : float
        Kích thước mỗi ảnh.

    random_sample : bool
        True  -> chọn ảnh ngẫu nhiên.
        False -> lấy ảnh theo thứ tự.

    title : str or None
        Tiêu đề.
    """

    extensions = (
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".bmp"
    )

    # ============================================
    # Lấy danh sách ảnh
    # ============================================

    image_files = sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(extensions)
    ])

    print(f"Folder: {folder}")
    print(f"Total images: {len(image_files)}")

    if len(image_files) == 0:
        print("Không tìm thấy ảnh.")
        return

    # ============================================
    # Số ảnh cần hiển thị
    # ============================================

    n = min(
        n_images,
        len(image_files)
    )

    # ============================================
    # Chọn ảnh
    # ============================================

    if random_sample:

        import random

        selected_files = random.sample(
            image_files,
            n
        )

    else:

        selected_files = image_files[:n]

    # ============================================
    # Đảm bảo cols hợp lệ
    # ============================================

    cols = max(
        1,
        min(cols, n)
    )

    # ============================================
    # Tính số hàng
    # ============================================

    rows = math.ceil(
        n / cols
    )

    # ============================================
    # Tạo figure
    # ============================================

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(
            figsize_per_image * cols,
            figsize_per_image * rows
        )
    )

    # Đưa axes về dạng 1D
    axes = (
        axes.flatten()
        if hasattr(axes, "flatten")
        else [axes]
    )

    # ============================================
    # Hiển thị ảnh
    # ============================================

    for i, filename in enumerate(
        selected_files
    ):

        path = os.path.join(
            folder,
            filename
        )

        try:

            img = Image.open(
                path
            ).convert("RGB")

            axes[i].imshow(img)

            axes[i].axis("off")

            axes[i].set_title(
                f"{i + 1}",
                fontsize=10
            )

        except Exception as e:

            print(
                f"Cannot read {path}: {e}"
            )

            axes[i].axis("off")

    # ============================================
    # Ẩn các ô thừa
    # ============================================

    for i in range(
        n,
        len(axes)
    ):
        axes[i].axis("off")

    # ============================================
    # Title
    # ============================================

    if title is not None:

        fig.suptitle(
            title,
            fontsize=14,
            fontweight="bold"
        )

    plt.tight_layout()

    plt.show()

In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image


def visualize_paths(
    paths,
    n_images=None,
    cols=8,
    figsize_per_image=3,
    random_sample=False
):
    """
    Visualize images directly from a list of image paths.

    Parameters
    ----------
    paths : list[str]
        Danh sách đường dẫn ảnh.

    n_images : int or None
        Số ảnh muốn hiển thị.
        None -> hiển thị tất cả.

    cols : int
        Số ảnh trên mỗi hàng.

    figsize_per_image : float
        Kích thước mỗi ảnh.

    random_sample : bool
        True -> chọn ảnh ngẫu nhiên.
        False -> lấy theo thứ tự trong paths.
    """

    paths = list(paths)

    if len(paths) == 0:
        print("Danh sách paths rỗng.")
        return

    # ==================================================
    # Chọn ảnh
    # ==================================================

    if n_images is None:
        n_images = len(paths)

    n_images = min(
        n_images,
        len(paths)
    )

    if random_sample:
        import random

        selected_paths = random.sample(
            paths,
            n_images
        )
    else:
        selected_paths = paths[:n_images]

    # ==================================================
    # Grid
    # ==================================================

    rows = math.ceil(
        n_images / cols
    )

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(
            cols * figsize_per_image,
            rows * figsize_per_image
        )
    )

    # Flatten axes để dễ xử lý
    axes = axes.flatten() if hasattr(
        axes,
        "flatten"
    ) else [axes]

    # ==================================================
    # Hiển thị ảnh
    # ==================================================

    for i, path in enumerate(
        selected_paths
    ):

        try:

            img = Image.open(
                path
            ).convert("RGB")

            axes[i].imshow(img)

            axes[i].set_title(
                f"{i + 1}",
                fontsize=10
            )

            axes[i].axis("off")

        except Exception as e:

            print(
                f"Cannot read: {path}"
            )

            print(e)

            axes[i].axis("off")

    # ==================================================
    # Ẩn ô thừa
    # ==================================================

    for i in range(
        n_images,
        len(axes)
    ):

        axes[i].axis("off")

    plt.tight_layout()

    plt.show()

In [ ]:
paths_dict['Annual Crop Land']

In [ ]:
visualize_paths(
    paths_dict['Annual Crop Land'],
    n_images=16,
    cols=8
)

In [ ]:
visualize_folder(
    folder="/content/Annual Crop Land",
    n_images=32,
    cols = 8,
    title="IsSynth  - Annual"
)

## 14. Check generated image counts

In [ ]:
!zip -r /content/disef_synth_64.zip /content/SPC/out_qwen

In [ ]:
counts = {}

for class_name in sorted(os.listdir(out_root)):
    class_path = os.path.join(
        out_root,
        class_name
    )

    if not os.path.isdir(class_path):
        continue

    counts[class_name] = len([
        x for x in os.listdir(class_path)
        if x.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ])

for class_name, count in counts.items():
    print(f"{class_name}: {count}")

print("Total generated:", sum(counts.values()))
